# Fase 3: Deteksi Anomali (Penjaga Garis Depan)

## Tujuan
Membuat alarm dini menggunakan Unsupervised Learning (Isolation Forest) untuk mendeteksi perilaku sensor yang tidak wajar.

## Langkah-langkah
- 3.1: Normalisasi Data (StandardScaler / MinMaxScaler)
- 3.2: Training Isolation Forest
- 3.3: Penerapan Aturan Bisnis Rolling Window (3 jam berturut-turut)
- 3.4: Evaluasi & Visualisasi Deteksi Anomali
- 3.5: Ekspor Model (anomaly_detector.pkl)

In [ ]:
import pandas as pd

print("Melakukan Sanity Check pada Data Ekspor (Fase 3)...\n")

# Memuat data hasil Fase 2 dengan menjadikan timestamp sebagai index bertipe datetime
df_ready = pd.read_csv('../data/processed/sensor_features_engineered.csv', index_col='timestamp', parse_dates=True)

# Validasi Dimensi Data
print(f"Bentuk Dataset (Shape): {df_ready.shape}")

# Validasi Kekosongan Data (Tidak boleh ada NaN)
max_nan = df_ready.isnull().sum().max()
print(f"Jumlah nilai NaN terbanyak pada salah satu kolom: {max_nan}")

# Menampilkan 5 baris pertama
print("\nSekilas isi df_ready:")
display(df_ready.head())

In [ ]:
# --- 3.1 Persiapan Data & Normalisasi ---

# 1. Buang sisa baris yang memiliki nilai kosong (NaN) pasca-rolling/FFT
df_ready.dropna(inplace=True)
print(f"Dimensi data setelah dropna: {df_ready.shape}")

# 2. Import StandardScaler
from sklearn.preprocessing import StandardScaler

# 3. Pisahkan label jawaban (Simpan kolom 'failure' ke variabel y)
# Kita pastikan kita tidak terkena error jika sewaktu-waktu kolomnya berganti nama.
if 'failure' in df_ready.columns:
    y = df_ready['failure']
else:
    y = None
    print("Himbauan: Kolom 'failure' tidak ditemukan. Mode Unsupervised murni.")

# 4. Buat variabel fitur X dengan membuang kolom target dan kolom teks ('machine_id')
cols_to_drop = ['machine_id']
if 'failure' in df_ready.columns:
    cols_to_drop.append('failure')
    
X = df_ready.drop(columns=[col for col in cols_to_drop if col in df_ready.columns])

# 5. Inisialisasi StandardScaler dan Lakukan Normalisasi pada X
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Mengubah array X_scaled menjadi DataFrame untuk visualisasi hasil
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print("\n--- Validasi Tabel Hasil Normalisasi (StandardScaler) ---")
display(X_scaled_df.head(5))

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

# --- 3.2 Training Isolation Forest ---
print("Memulai pelatihan model pendeteksi anomali Isolation Forest...")

# 1 & 2. Inisialisasi model isolation forest
# contamination=0.01 berarti kita berasumsi maksimal 1% data operasional adalah anomali
model_if = IsolationForest(contamination=0.01, n_estimators=100, random_state=42)

# 3. Latih model sekaligus memprediksi suspek anomali
# Ini dapat memakan waktu beberapa detik karena memproses ~100.000 baris x banyak fitur
preds = model_if.fit_predict(X_scaled)

# 4. Menerjemahkan output model (-1 = Anomali, 1 = Normal) menjadi boolean numerik (1 = Anomali, 0 = Normal)
df_ready['is_anomaly'] = np.where(preds == -1, 1, 0)
print("Pelatihan selesai!\n")

# --- 3.3 Penerapan Aturan Bisnis Rolling Window (3 jam berturut-turut) ---
print("Memfilter status anomali dengan Aturan Bisnis (Alarm 3 Jam)...\n")

# 5. Logika Aturan Bisnis
# Menerapkan jendela waktu berjalan (rolling window) selama 3 jam, lalu filter 
# apabila nilainya berjumlah 3 (berarti 3 jam berturut-turut mesin anomali)
df_ready['is_rolling_anomaly'] = df_ready['is_anomaly'].rolling(window=3).sum() >= 3
df_ready['is_rolling_anomaly'] = df_ready['is_rolling_anomaly'].astype(int) # Jadikan 1/0 agar rapi

# 6. Cetak Value Counts untuk Analisis Alarm
print("--- Distribusi Sensor Anomali (Mentah) vs Alarm (Filter) ---")
print("1. Anomali Mentah (is_anomaly):")
display(df_ready['is_anomaly'].value_counts())

print("\n2. Alarm Sesungguhnya (is_rolling_anomaly):")
display(df_ready['is_rolling_anomaly'].value_counts())

print("\n✅ Isolasi Ekstrak Aturan Bisnis (Langkah 3.2 - 3.4) Selesai!")

In [ ]:
import matplotlib.pyplot as plt

# --- 3.5 Evaluasi Deteksi Anomali & Validasi Fisika ---
print("Memulai visualisasi validasi anomali...\n")

# 1. Cari tahu machine_id mana saja yang memicu alarm sesungguhnya
mesin_terindikasi = df_ready[df_ready['is_rolling_anomaly'] == 1]['machine_id'].unique()
print(f"Mesin yang memiliki alarm menyala: {mesin_terindikasi}")

if len(mesin_terindikasi) > 0:
    # Kita asumsikan kita mengambil satu sample mesin pertama yang rusak untuk diinvestigasi
    target_machine = mesin_terindikasi[0]
    print(f"\nMenggambar plot investigasi untuk mesin: {target_machine}\n")

    # 2. Filter data hanya untuk mesin target tersebut
    df_machine = df_ready[df_ready['machine_id'] == target_machine]

    # Identifikasi titik-titik krusial
    anomaly_points = df_machine[df_machine['is_rolling_anomaly'] == 1]
    
    failure_points = pd.DataFrame() # Kosong sebagai default jika tidak ada kegagalan nyata
    if 'failure' in df_machine.columns:
        failure_points = df_machine[df_machine['failure'] == 1]

    # 3. Membuat visualisasi plot 2 sumbu: Temperature dan Vibration
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    fig.suptitle(f'Sanity Check: Evaluasi Fisika Isolation Forest pada Mesin {target_machine}', fontsize=16)

    # === SUBPLOT 1: TEMPERATURE ===
    ax1.plot(df_machine.index, df_machine['temperature'], label='Suhu Normal (Celcius)', color='blue', alpha=0.6)
    # 4. Berikan titik MERAH TEBAL saat Alarm Anomali berjalan
    ax1.scatter(anomaly_points.index, anomaly_points['temperature'], color='red', s=100, label='Alarm Anomali Aktif', zorder=5)
    # 5. Garis Penanda Kegagalan Nyata (Baseline Kebenaran)
    if not failure_points.empty:
        for failure_time in failure_points.index:
            ax1.axvline(x=failure_time, color='black', linestyle='--', linewidth=2, label='Mesin Rusak Nyata (Failure=1)' if failure_time == failure_points.index[0] else "")
    ax1.set_ylabel('Temperature')
    ax1.legend()
    ax1.grid(True)

    # === SUBPLOT 2: VIBRATION ===
    ax2.plot(df_machine.index, df_machine['vibration'], label='Getaran Normal (mm/s)', color='green', alpha=0.6)
    # 4. Berikan titik MERAH TEBAL saat Alarm Anomali berjalan
    ax2.scatter(anomaly_points.index, anomaly_points['vibration'], color='red', s=100, label='Alarm Anomali Aktif', zorder=5)
    # 5. Garis Penanda Kegagalan Nyata (Baseline Kebenaran)
    if not failure_points.empty:
        for failure_time in failure_points.index:
            ax2.axvline(x=failure_time, color='black', linestyle='--', linewidth=2, label='Mesin Rusak Nyata (Failure=1)' if failure_time == failure_points.index[0] else "")
    ax2.set_xlabel('Timestamp')
    ax2.set_ylabel('Vibration')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()
else:
    print("TIDAK ADA mesin yang terindikasi anomali (alarm tidak pernah menyala).")


In [ ]:
import os
import joblib

# --- Ekspor Model --- 
print("Mempersiapkan ekspor model Isolation Forest...")

# 1. Tentukan path direktori penyimpanan model target
model_dir = '../models/'

# 2. Jika direktori 'models' belum ada, ciptakan secara otomatis (mirip mkdir -p)
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, 'anomaly_detector.pkl')

# 3. Lakukan proses Pickling (Dump) arsitektur model beserta otak (weights)-nya ke format biner (.pkl)
joblib.dump(model_if, model_path)

# 4. Konfirmasi Kesuksesan
print(f"✅ Model Isolation Forest berhasil diekspor ke {model_path}!")
print("Model ini sekarang sudah beku (frozen) dan siap dipanggil (loaded) kapanpun di sistem produksi.")